# Ablation Study — NASA Battery B0005

Each component used by the better-performing battery models is switched on one at a time.
**Protocol unchanged:** chronological split (train `cycle < 50`, test `cycle >= 50`), scaler
fitted on the training cycles only, sliding window of 10 cycles.

Results are reported on the **SOH scale (0–1)** so they are directly comparable with the
published battery literature, and converted back to Ah at the end.

| ID | Configuration | Component added | Source of the idea |
|----|---------------|-----------------|--------------------|
| A0 | Baseline (as reported, 0.047 Ah = 0.0253 SOH) | capacity only, unidirectional LSTM | our published result |
| A1 | + health indicators | 9 indicators engineered from each discharge curve | Tian et al. [53] (0.00481) |
| A2 | + bidirectional | BiLSTM | Tian [53], Gao [new] |
| A3 | + attention pooling | attention over the sequence | CNN-BiLSTM-**AM** [53] |
| A4 | + multi-scale CNN | parallel kernels 3 / 5 / 7 | MSTSDN [58] |
| A5 | + ensemble | average of 5 seeds | our own reconstruction |

Run from the folder containing `B0005.mat`.

In [1]:
import os, json, gc, time
import numpy as np, pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, Masking, Softmax,
                                     Multiply, Lambda, Flatten, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TF', tf.__version__, '| GPU', tf.config.list_physical_devices('GPU'))

TF 2.10.0 | GPU [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
NAME, SLUG = 'NASA Battery B0005', 'battery_b0005'
MAT_PATH    = 'B0005.mat'
WINDOW      = 10
SPLIT_CYCLE = 50
N_ENSEMBLE  = 5
EPOCHS, BATCH, PATIENCE = 200, 16, 20
MASKING     = False
BASELINE_RMSE = 0.0253      # 0.047 Ah expressed on the SOH scale

In [4]:
from scipy.io import loadmat
mat = loadmat(MAT_PATH); cycles = mat['B0005'][0, 0]['cycle'][0]
rows, n = [], 0
for i in range(len(cycles)):
    r = cycles[i]
    if r['type'][0] != 'discharge':
        continue
    n += 1; d = r['data'][0][0]
    V = d['Voltage_measured'][0].astype(float); I = d['Current_measured'][0].astype(float)
    T = d['Temperature_measured'][0].astype(float); t = d['Time'][0].astype(float)
    tt = lambda vth: (float(t[np.where(V <= vth)[0][0]]) if len(np.where(V <= vth)[0]) else float(t[-1]))
    rows.append(dict(cycle=n, capacity=float(d['Capacity'][0][0]),
                     discharge_time=float(t[-1]), t_to_3p8V=tt(3.8), t_to_3p5V=tt(3.5),
                     mean_voltage=float(V.mean()), min_voltage=float(V.min()),
                     mean_temp=float(T.mean()), max_temp=float(T.max()),
                     temp_rise=float(T.max() - T[0])))
HI = pd.DataFrame(rows)
C0 = HI['capacity'].iloc[0]
HI['SOH'] = HI['capacity'] / C0
BASE_COLS = ['capacity']                                   # A0: capacity only
FULL_COLS = [c for c in HI.columns if c not in ('cycle', 'SOH')]   # A1+: engineered indicators
print('cycles %d | initial capacity %.4f Ah' % (len(HI), C0))
print('\ncorrelation of each indicator with SOH:')
print(HI[FULL_COLS].corrwith(HI['SOH']).round(3).to_string())

cycles 168 | initial capacity 1.8565 Ah

correlation of each indicator with SOH:
capacity          1.000
discharge_time    0.976
t_to_3p8V         0.987
t_to_3p5V         0.998
mean_voltage      0.982
min_voltage      -0.487
mean_temp        -0.810
max_temp         -0.935
temp_rise        -0.979


In [5]:
def load(variant):
    """variant = 'base' (capacity only) or 'full' (engineered health indicators)."""
    cols = BASE_COLS if variant == 'base' else FULL_COLS
    sc = MinMaxScaler().fit(HI[HI.cycle < SPLIT_CYCLE][cols].values)
    Fm = sc.transform(HI[cols].values).astype(np.float32)
    S  = HI['SOH'].values.astype(np.float32)
    Xa, ya, Xt, yt = [], [], [], []
    for e in range(WINDOW, len(HI)):
        seq = Fm[e - WINDOW:e]
        if HI['cycle'].iloc[e] < SPLIT_CYCLE:
            Xa.append(seq); ya.append(S[e])
        else:
            Xt.append(seq); yt.append(S[e])
    Xa = np.asarray(Xa, np.float32); ya = np.asarray(ya, np.float32)
    Xt = np.asarray(Xt, np.float32); yt = np.asarray(yt, np.float32)
    cut = int(0.8 * len(Xa))
    return Xa[:cut], ya[:cut], Xa[cut:], ya[cut:], Xt, yt

In [6]:
def attention_pool(seq):
    """Additive attention over the sequence, replacing 'use the last hidden state'."""
    score = Dense(1, activation='tanh')(seq)
    w = Softmax(axis=1)(score)
    return Lambda(lambda x: K.sum(x, axis=1))(Multiply()([seq, w]))

def build_model(shape, bidirectional=False, attention=False, multiscale=False,
                masking=False, units=64, filters=64, lr=1e-3):
    """Parallel CNN-LSTM. Each flag switches on one ablated component."""
    inp = Input(shape=shape)
    x = Masking(mask_value=-99.)(inp) if masking else inp
    # convolutional branch
    if multiscale:
        c = Concatenate()([Conv1D(filters, k, padding='same', activation='relu')(x)
                           for k in (3, 5, 7)])
    else:
        c = Conv1D(filters, 3, padding='same', activation='relu')(x)
    c = Dropout(0.1)(c)
    if int(shape[0]) > 3:
        c = MaxPooling1D(2, padding='same')(c)
    c = Conv1D(32, 3, padding='same', activation='relu')(c)
    c = GlobalAveragePooling1D()(c)
    # recurrent branch
    r = Bidirectional(LSTM(units, return_sequences=attention))(x) if bidirectional \
        else LSTM(units, return_sequences=attention)(x)
    if attention:
        r = attention_pool(r)
    # head
    h = Concatenate()([c, r])
    h = Dense(128, activation='relu')(h); h = Dropout(0.2)(h)
    h = Dense(64, activation='relu')(h)
    m = Model(inp, Dense(1, activation='linear')(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='mse', metrics=['mae'])
    return m

In [7]:
def train_eval(cfg, data, seeds=1):
    Xtr, ytr, Xva, yva, Xte, yte = data
    preds, singles, params = [], [], None
    for s in range(seeds):
        tf.random.set_seed(SEED + s); np.random.seed(SEED + s)
        m = build_model(Xtr.shape[1:], masking=MASKING, **cfg)
        params = m.count_params()
        m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=BATCH, verbose=0,
              callbacks=[EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
                         ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)])
        p = m.predict(Xte, batch_size=512, verbose=0).flatten()
        preds.append(p); singles.append(float(np.sqrt(mean_squared_error(yte, p))))
        K.clear_session()
    ens = np.mean(preds, axis=0)
    return dict(rmse=float(np.sqrt(mean_squared_error(yte, ens))),
                mae=float(mean_absolute_error(yte, ens)), r2=float(r2_score(yte, ens)),
                seeds=seeds, per_seed=[round(x, 4) for x in singles], params=int(params))

def run_ladder(ladder, loader):
    cache, results = {}, []
    t0 = time.time()
    for rid, label, key, cfg, seeds in ladder:
        if key not in cache:
            print('loading data variant:', key, '...')
            cache[key] = loader(key)
            print('   train', cache[key][0].shape, '| test', cache[key][4].shape)
        print('--- %s: %s (seeds=%d) ---' % (rid, label, seeds))
        r = train_eval(cfg, cache[key], seeds=seeds)
        r.update(id=rid, config=label, variant=str(key))
        results.append(r)
        print('    RMSE %.4f | MAE %.4f | R2 %.4f | params %s'
              % (r['rmse'], r['mae'], r['r2'], format(r['params'], ',')))
    print('\ntotal %.0f s' % (time.time() - t0))
    return results

In [8]:
LADDER = [
    ("A0", "Baseline (capacity only)", 'base', dict(bidirectional=False, attention=False, multiscale=False), 1),
    ("A1", "+ health indicators",      'full', dict(bidirectional=False, attention=False, multiscale=False), 1),
    ("A2", "+ bidirectional",          'full', dict(bidirectional=True,  attention=False, multiscale=False), 1),
    ("A3", "+ attention pooling",      'full', dict(bidirectional=True,  attention=True,  multiscale=False), 1),
    ("A4", "+ multi-scale CNN",        'full', dict(bidirectional=True,  attention=True,  multiscale=True),  1),
    ("A5", "+ ensemble",               'full', dict(bidirectional=True,  attention=True,  multiscale=True),  N_ENSEMBLE),
]
results = run_ladder(LADDER, load)

loading data variant: base ...
   train (31, 10, 1) | test (119, 10, 1)
--- A0: Baseline (capacity only) (seeds=1) ---
    RMSE 0.4063 | MAE 0.3413 | R2 -30.0714 | params 44,065
loading data variant: full ...
   train (31, 10, 9) | test (119, 10, 9)
--- A1: + health indicators (seeds=1) ---
    RMSE 0.5159 | MAE 0.4951 | R2 -49.0960 | params 47,649
--- A2: + bidirectional (seeds=1) ---
    RMSE 0.7268 | MAE 0.6845 | R2 -98.4245 | params 74,785
--- A3: + attention pooling (seeds=1) ---
    RMSE 0.4998 | MAE 0.4701 | R2 -46.0247 | params 74,914
--- A4: + multi-scale CNN (seeds=1) ---
    RMSE 0.7567 | MAE 0.7142 | R2 -106.7694 | params 94,242
--- A5: + ensemble (seeds=5) ---
    RMSE 0.4716 | MAE 0.4501 | R2 -40.8672 | params 94,242

total 78 s


In [9]:
tbl = pd.DataFrame(results)[['id','config','variant','rmse','mae','r2','params','seeds']]
tbl['delta_vs_prev'] = tbl['rmse'].diff().round(4)
tbl['delta_vs_A0']   = (tbl['rmse'] - tbl['rmse'].iloc[0]).round(4)
tbl['improve_%']     = ((tbl['rmse'].iloc[0] - tbl['rmse']) / tbl['rmse'].iloc[0] * 100).round(2)
pd.set_option('display.width', 170)
print('DATASET: %s   published baseline RMSE = %s\n' % (NAME, BASELINE_RMSE))
print(tbl.to_string(index=False))
best = tbl.loc[tbl['rmse'].idxmin()]
print('\nBest: %s (%s) at RMSE %.4f  |  vs published baseline %+.4f'
      % (best['id'], best['config'], best['rmse'], best['rmse'] - BASELINE_RMSE))
print('\nSOH -> capacity: best %.4f SOH = %.4f Ah   (published baseline 0.047 Ah)'
      % (best['rmse'], best['rmse'] * C0))

DATASET: NASA Battery B0005   published baseline RMSE = 0.0253

id                   config variant     rmse      mae          r2  params  seeds  delta_vs_prev  delta_vs_A0  improve_%
A0 Baseline (capacity only)    base 0.406286 0.341329  -30.071430   44065      1            NaN       0.0000       0.00
A1      + health indicators    full 0.515885 0.495123  -49.095999   47649      1         0.1096       0.1096     -26.98
A2          + bidirectional    full 0.726772 0.684482  -98.424505   74785      1         0.2109       0.3205     -78.88
A3      + attention pooling    full 0.499821 0.470061  -46.024690   74914      1        -0.2270       0.0935     -23.02
A4        + multi-scale CNN    full 0.756658 0.714173 -106.769421   94242      1         0.2568       0.3504     -86.24
A5               + ensemble    full 0.471616 0.450109  -40.867189   94242      5        -0.2850       0.0653     -16.08

Best: A0 (Baseline (capacity only)) at RMSE 0.4063  |  vs published baseline +0.3810

SOH -> ca

In [10]:
out = {'dataset': NAME, 'published_baseline_rmse': BASELINE_RMSE,
       'ablation': results, 'best': {'id': str(best['id']), 'rmse': float(best['rmse'])}}
try:
    out['pooling_ablation'] = pool_rows
except NameError:
    pass
json.dump(out, open('ablation_%s_results.json' % SLUG, 'w'), indent=2)
tbl.to_csv('ablation_%s_table.csv' % SLUG, index=False)
print('saved ablation_%s_results.json and ablation_%s_table.csv' % (SLUG, SLUG))

saved ablation_battery_b0005_results.json and ablation_battery_b0005_table.csv
